# Распознавание эмоций по аудио-сегментам (HuBERT, RU)

Этот ноутбук: 
- читает CSV сегментов после diarization;
- валидирует сегменты: отбрасывает длительность < 1.5с;
- объединяет соседние сегменты одного спикера при паузе между ними ≤ 0.5с;
- вырезает аудиофрагменты из исходного файла звонка;
- приводит фрагменты к `wav / mono / 16 kHz`;
- запускает **ТОЛЬКО** модель `xbgoose/hubert-large-speech-emotion-recognition-russian-dusha-finetuned`;
- возвращает JSON с вероятностями эмоций для каждого итогового сегмента.

Ограничения: никаких ASR/текста, никакой семантической интерпретации, каждый сегмент анализируется независимо.

## Установка зависимостей

Если пакеты не установлены в текущем kernel, выполните следующую ячейку.

Важно: для вырезки аудио используется **ffmpeg** (это не Python-пакет). Установите ffmpeg отдельно и убедитесь, что команда `ffmpeg` доступна в терминале.

In [ ]:
# Если нужно — раскомментируйте и выполните
# %pip install -q pandas numpy soundfile transformers torch tqdm

In [1]:
from __future__ import annotations

import io
import json
import os
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import soundfile as sf
import torch
from transformers import HubertForSequenceClassification, Wav2Vec2FeatureExtractor

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = lambda x, **_: x  # type: ignore

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", 120)

## Конфигурация

В `results_segments.csv` пути к аудио могут быть абсолютными (например, `Z:\...`). Если этот диск не смонтирован в текущей среде, задайте `PATH_REMAP` как соответствие префиксов путей.

In [2]:
# Вход/выход
CSV_PATH = Path("data/diarization_experiment/20260301_202341/results_segments.csv")
OUTPUT_JSON_PATH = Path(
    "data/diarization_experiment/20260301_202341/emotion_segments_hubert_dusha.json"
)

# Правила валидации
MIN_DURATION_S = 1.5
MAX_GAP_S = 0.5

# Аудио
TARGET_SAMPLE_RATE = 16_000

# Модель (строго заданная по требованию)
MODEL_ID = "xbgoose/hubert-large-speech-emotion-recognition-russian-dusha-finetuned"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Доступно GPU: {torch.cuda.device_count()}")
else:
    print("CUDA не найдена, будет использоваться CPU")

# (Опционально) Ремап путей: {исходный_префикс: новый_префикс}
# Пример: PATH_REMAP = {r'Z:\calls\dataset': r'D:\calls\dataset'}
PATH_REMAP: Dict[str, str] = {}

Используется устройство: cuda
GPU: NVIDIA GeForce RTX 4060
Доступно GPU: 1


In [3]:
def resolve_audio_path(raw_path: str, path_remap: Dict[str, str]) -> Path:
    """Пытается найти файл по raw_path; при необходимости применяет ремап префикса."""
    p = Path(raw_path)
    if p.exists():
        return p

    raw_norm = os.path.normcase(os.path.normpath(raw_path))
    for src_prefix, dst_prefix in path_remap.items():
        src_norm = os.path.normcase(os.path.normpath(src_prefix))
        if raw_norm.startswith(src_norm):
            rel = raw_path[len(src_prefix) :].lstrip("\\/")
            candidate = Path(dst_prefix) / rel
            if candidate.exists():
                return candidate

    return p

## 1) Загрузка CSV и валидация/слияние сегментов

In [4]:
df = pd.read_csv(CSV_PATH)
expected_cols = {"model", "id", "path", "start", "end", "speaker", "duration"}
missing = expected_cols - set(df.columns)
if missing:
    raise ValueError(f"В CSV нет обязательных колонок: {sorted(missing)}")

# Приведение типов
df["start"] = df["start"].astype(float)
df["end"] = df["end"].astype(float)
df["duration"] = df["duration"].astype(float)
df["id"] = df["id"].astype(str)
df["speaker"] = df["speaker"].astype(str)
df["path"] = df["path"].astype(str)

# Фильтр по длительности (< 1.5с — игнорируем)
df_valid = df[df["duration"] >= MIN_DURATION_S].copy()

print("Всего строк в CSV:", len(df))
print("После фильтра duration >=", MIN_DURATION_S, ":", len(df_valid))
df_valid.head(10)

Всего строк в CSV: 1141
После фильтра duration >= 1.5 : 433


,model,id,path,start,end,speaker,duration
2,pyannote/speaker-diarization-3.1,9450df88675f07c12142bffeeed0a9e1bedce4e7,Z:\calls\dataset\2025\12\18\1766039757.126564.wav,2.292219,3.945969,SPEAKER_01,1.653750
7,pyannote/speaker-diarization-3.1,9450df88675f07c12142bffeeed0a9e1bedce4e7,Z:\calls\dataset\2025\12\18\1766039757.126564.wav,8.485344,10.442844,SPEAKER_00,1.957500
11,pyannote/speaker-diarization-3.1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,1.600344,4.064094,SPEAKER_00,2.463750
12,pyannote/speaker-diarization-3.1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,4.924719,9.227844,SPEAKER_00,4.303125
20,pyannote/speaker-diarization-3.1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,17.310969,21.344094,SPEAKER_00,4.033125
21,pyannote/speaker-diarization-3.1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,21.985344,25.630344,SPEAKER_01,3.645000
22,pyannote/speaker-diarization-3.1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,25.934094,27.992844,SPEAKER_01,2.058750
24,pyannote/speaker-diarization-3.1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,33.662844,35.502219,SPEAKER_01,1.839375
26,pyannote/speaker-diarization-3.1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,42.353469,47.483469,SPEAKER_01,5.130000
30,pyannote/speaker-diarization-3.1,23246d7975ff89d56bf38458e614b716512ba06d,Z:\calls\dataset\2025\12\22\1766391440.276622.wav,50.132844,53.592219,SPEAKER_01,3.459375


In [5]:
@dataclass
class Segment:
    model: str
    call_id: str
    path: str
    speaker: str
    start: float
    end: float

    @property
    def duration(self) -> float:
        return float(self.end - self.start)


def merge_adjacent_segments(
    segments_df: pd.DataFrame,
    max_gap_s: float,
) -> pd.DataFrame:
    """Сливает соседние сегменты одного спикера внутри звонка при паузе <= max_gap_s.

    Предполагается, что короткие сегменты уже отфильтрованы.
    """
    segs = segments_df.sort_values(["id", "speaker", "start", "end"]).reset_index(
        drop=True
    )

    merged: List[Segment] = []
    for row in segs.itertuples(index=False):
        current = Segment(
            model=getattr(row, "model"),
            call_id=getattr(row, "id"),
            path=getattr(row, "path"),
            speaker=getattr(row, "speaker"),
            start=float(getattr(row, "start")),
            end=float(getattr(row, "end")),
        )

        if not merged:
            merged.append(current)
            continue

        prev = merged[-1]
        same_group = (prev.call_id == current.call_id) and (
            prev.speaker == current.speaker
        )
        same_path = prev.path == current.path
        gap = current.start - prev.end

        if same_group and same_path and gap <= max_gap_s:
            # Сливаем (включая перекрытия, когда gap < 0)
            prev.end = max(prev.end, current.end)
        else:
            merged.append(current)

    out = pd.DataFrame(
        [
            {
                "model": s.model,
                "id": s.call_id,
                "path": s.path,
                "start": s.start,
                "end": s.end,
                "speaker": s.speaker,
                "duration": s.duration,
            }
            for s in merged
        ]
    )
    return out


df_merged = merge_adjacent_segments(df_valid, max_gap_s=MAX_GAP_S)
print("После слияния:", len(df_merged))
df_merged.head(10)

После слияния: 354


,model,id,path,start,end,speaker,duration
0,pyannote/speaker-diarization-3.1,043e79e1e4cd518e69aa8613e5e40ab50376289b,Z:\calls\dataset\2026\01\14\1768398663.65115.wav,0.520344,2.714094,SPEAKER_00,2.193750
1,pyannote/speaker-diarization-3.1,06370c52b6c1e38ad1f95dd4610fa49a2c946562,Z:\calls\dataset\2025\12\22\1766395111.284523.wav,2.224719,5.042844,SPEAKER_00,2.818125
2,pyannote/speaker-diarization-3.1,06370c52b6c1e38ad1f95dd4610fa49a2c946562,Z:\calls\dataset\2025\12\22\1766395111.284523.wav,9.464094,11.860344,SPEAKER_00,2.396250
3,pyannote/speaker-diarization-3.1,06370c52b6c1e38ad1f95dd4610fa49a2c946562,Z:\calls\dataset\2025\12\22\1766395111.284523.wav,14.678469,16.365969,SPEAKER_00,1.687500
4,pyannote/speaker-diarization-3.1,06370c52b6c1e38ad1f95dd4610fa49a2c946562,Z:\calls\dataset\2025\12\22\1766395111.284523.wav,19.133469,23.909094,SPEAKER_00,4.775625
5,pyannote/speaker-diarization-3.1,06370c52b6c1e38ad1f95dd4610fa49a2c946562,Z:\calls\dataset\2025\12\22\1766395111.284523.wav,31.941594,33.797844,SPEAKER_00,1.856250
6,pyannote/speaker-diarization-3.1,06370c52b6c1e38ad1f95dd4610fa49a2c946562,Z:\calls\dataset\2025\12\22\1766395111.284523.wav,34.472844,40.598469,SPEAKER_00,6.125625
7,pyannote/speaker-diarization-3.1,06370c52b6c1e38ad1f95dd4610fa49a2c946562,Z:\calls\dataset\2025\12\22\1766395111.284523.wav,41.594094,44.614719,SPEAKER_00,3.020625
8,pyannote/speaker-diarization-3.1,06370c52b6c1e38ad1f95dd4610fa49a2c946562,Z:\calls\dataset\2025\12\22\1766395111.284523.wav,25.529094,29.174094,SPEAKER_01,3.645000
9,pyannote/speaker-diarization-3.1,077275e8d5b6b39ffe4f04cf96133a82d7ed5900,Z:\calls\dataset\2025\12\29\1767002810.605617.wav,2.005344,4.739094,SPEAKER_00,2.733750


## 2) Вырезка аудио-сегмента и приведение к wav/mono/16kHz

Ниже функция, которая **через ffmpeg** вырезает нужный интервал и сразу приводит его к `wav / mono / 16 kHz`.

> Требование: бинарник `ffmpeg` должен быть доступен в `PATH` (или укажите полный путь к `ffmpeg.exe` в функции).

In [6]:
def load_audio_segment_16k_mono(
    audio_path: Path,
    start_s: float,
    end_s: float,
    target_sr: int = 16_000,
    dtype: str = "float32",
    ffmpeg_bin: str = "ffmpeg",
) -> np.ndarray:
    """Вырезает [start_s, end_s] через ffmpeg и возвращает waveform float32 (mono, target_sr)."""
    if end_s <= start_s:
        raise ValueError(f"Некорректный интервал: start={start_s}, end={end_s}")

    duration_s = float(end_s - start_s)
    cmd = [
        ffmpeg_bin,
        "-hide_banner",
        "-loglevel",
        "error",
        "-nostdin",
        "-ss",
        str(start_s),
        "-t",
        str(duration_s),
        "-i",
        str(audio_path),
        "-ac",
        "1",
        "-ar",
        str(int(target_sr)),
        "-f",
        "wav",
        "pipe:1",
    ]

    try:
        proc = subprocess.run(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=False,
        )
    except FileNotFoundError as e:
        raise RuntimeError(
            "ffmpeg не найден. Установите ffmpeg и добавьте в PATH, "
            "или передайте полный путь к ffmpeg.exe в параметре ffmpeg_bin."
        ) from e

    if proc.returncode != 0:
        err = proc.stderr.decode("utf-8", errors="ignore")
        raise RuntimeError(
            f"ffmpeg завершился с ошибкой (code={proc.returncode}):\n{err}"
        )

    if not proc.stdout:
        raise RuntimeError("ffmpeg вернул пустой вывод")

    with sf.SoundFile(io.BytesIO(proc.stdout), mode="r") as f:
        sr = int(f.samplerate)
        channels = int(f.channels)
        audio = f.read(dtype=dtype, always_2d=False)

    if sr != int(target_sr):
        raise RuntimeError(f"Ожидали sample_rate={target_sr}, получили {sr}")
    if channels != 1:
        # На всякий случай, хотя -ac 1 должен гарантировать mono
        audio = np.mean(audio, axis=1).astype(np.float32, copy=False)
    else:
        audio = np.asarray(audio, dtype=np.float32)

    if audio.size == 0:
        raise RuntimeError("Пустой аудиофрагмент после ffmpeg")
    return audio

## 3) Инференс эмоций строго через HuBERT-модель

Инференс делается напрямую через `HubertForSequenceClassification` + `Wav2Vec2FeatureExtractor` для модели `xbgoose/hubert-large-speech-emotion-recognition-russian-dusha-finetuned` (softmax по logits).

In [7]:
# Загрузка препроцессора и модели КЛАССИФИКАЦИИ эмоций (строго заданная модель)
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_ID)
model = HubertForSequenceClassification.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

id2label = getattr(model.config, "id2label", None) or {}
label2id = getattr(model.config, "label2id", None) or {}
num_labels = int(getattr(model.config, "num_labels", len(id2label) or 0) or 0)

print("Модель загружена:", MODEL_ID)
print("Устройство:", DEVICE)
print("num_labels:", num_labels)
print(
    "labels:",
    (
        [id2label.get(i, str(i)) for i in range(num_labels)]
        if num_labels
        else list(id2label.values())
    ),
)

Модель загружена: xbgoose/hubert-large-speech-emotion-recognition-russian-dusha-finetuned
Устройство: cuda
num_labels: 5
labels: ['neutral', 'angry', 'positive', 'sad', 'other']


## 4) Прогон по итоговым сегментам и экспорт JSON

Если часть аудиофайлов недоступна по указанному пути, добавьте `PATH_REMAP` в конфигурации выше.

In [8]:
MAX_MODEL_INPUT_SECONDS: float = 10.0


def infer_emotion_probs(wav16k: np.ndarray) -> Dict[str, float]:
    if wav16k.ndim != 1:
        wav16k = np.asarray(wav16k).reshape(-1)
    max_len = (
        int(feature_extractor.sampling_rate * MAX_MODEL_INPUT_SECONDS)
        if MAX_MODEL_INPUT_SECONDS
        else None
    )

    inputs = feature_extractor(
        wav16k,
        sampling_rate=feature_extractor.sampling_rate,
        return_tensors="pt",
        padding=True,
        truncation=True if max_len else False,
        max_length=max_len,
    )
    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits  # (1, num_labels)
        probs_t = torch.softmax(logits, dim=-1)[0].detach().cpu()
        probs = probs_t.numpy().astype(float).tolist()

    n = len(probs)
    if not getattr(model.config, "id2label", None):
        labels = [str(i) for i in range(n)]
    else:
        labels = [model.config.id2label.get(i, str(i)) for i in range(n)]
    return {labels[i]: float(probs[i]) for i in range(n)}


emotion_segments: List[Dict[str, Any]] = []
errors: List[Dict[str, Any]] = []

for row in tqdm(df_merged.itertuples(index=False), total=len(df_merged)):
    call_id = str(getattr(row, "id"))
    speaker = str(getattr(row, "speaker"))
    start = float(getattr(row, "start"))
    end = float(getattr(row, "end"))
    duration = float(getattr(row, "duration"))
    raw_path = str(getattr(row, "path"))

    audio_path = resolve_audio_path(raw_path, PATH_REMAP)
    if not audio_path.exists():
        errors.append(
            {
                "call_id": call_id,
                "speaker": speaker,
                "start": start,
                "end": end,
                "reason": "audio_file_not_found",
                "path": raw_path,
                "resolved_path": str(audio_path),
            }
        )
        continue

    try:
        wav16k = load_audio_segment_16k_mono(
            audio_path, start, end, target_sr=TARGET_SAMPLE_RATE
        )

        probs = infer_emotion_probs(wav16k)
        emotion_label = max(probs.items(), key=lambda kv: kv[1])[0]

        emotion_segments.append(
            {
                "call_id": call_id,
                "speaker": speaker,
                "start": start,
                "end": end,
                "duration": duration,
                "emotion_label": emotion_label,
                "emotion_probs": probs,
            }
        )
    except Exception as e:
        errors.append(
            {
                "call_id": call_id,
                "speaker": speaker,
                "start": start,
                "end": end,
                "reason": "inference_error",
                "path": raw_path,
                "resolved_path": str(audio_path),
                "error": repr(e),
            }
        )

result = {"emotion_segments": emotion_segments}

OUTPUT_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("Готово. emotion_segments:", len(emotion_segments))
print("Ошибок:", len(errors))
print("JSON сохранён в:", OUTPUT_JSON_PATH)

emotion_segments[:3]

  0%|          | 0/354 [00:00<?, ?it/s]

Готово. emotion_segments: 354
Ошибок: 0
JSON сохранён в: data\diarization_experiment\20260301_202341\emotion_segments_hubert_dusha.json


[{'call_id': '043e79e1e4cd518e69aa8613e5e40ab50376289b',
  'speaker': 'SPEAKER_00',
  'start': 0.52034375,
  'end': 2.71409375,
  'duration': 2.19375,
  'emotion_label': 'positive',
  'emotion_probs': {'neutral': 0.1535235345363617,
   'angry': 0.010831325314939022,
   'positive': 0.8306407928466797,
   'sad': 0.002894812962040305,
   'other': 0.0021095711272209883}},
 {'call_id': '06370c52b6c1e38ad1f95dd4610fa49a2c946562',
  'speaker': 'SPEAKER_00',
  'start': 2.22471875,
  'end': 5.042843750000001,
  'duration': 2.818125000000001,
  'emotion_label': 'positive',
  'emotion_probs': {'neutral': 0.35422825813293457,
   'angry': 0.0019391704117879272,
   'positive': 0.6406930088996887,
   'sad': 0.0023175147362053394,
   'other': 0.0008220080635510385}},
 {'call_id': '06370c52b6c1e38ad1f95dd4610fa49a2c946562',
  'speaker': 'SPEAKER_00',
  'start': 9.46409375,
  'end': 11.860343750000002,
  'duration': 2.396250000000002,
  'emotion_label': 'neutral',
  'emotion_probs': {'neutral': 0.562043

## (Опционально) Просмотр ошибок
Если `errors` не пуст, чаще всего это отсутствие исходного аудио по пути из CSV. В таком случае заполните `PATH_REMAP` и перезапустите прогон.

In [9]:
# Показать первые 20 ошибок
errors[:20]

[]